# Terputer transcriptomics research notebook

**Question:** Which metabolic and signaling responses to terpene exposure are reproducible, and which depend on cellular context or a specific target?

This notebook demonstrates Terputer's RNA-seq and Perturb-seq workflow with three frozen public inputs: processed linalool bulk RNA-seq (GSE136998), processed fungal terpene-exposure RNA-seq (GSE138291), and processed A549 genetic Perturb-seq signatures (Arc PIE Jiang). The analyses are descriptive. The samples do not establish cross-study terpene replication or target binding. Source URLs, revisions, checksums, study designs, and limitations are recorded below and in `research/transcriptomics/data/`.


In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
sys.path.insert(0, str(Path.cwd()))
from terputer_bio.transcriptomics import descriptive_fpkm_ratio
ROOT = Path.cwd()
DATA = ROOT / 'research' / 'transcriptomics' / 'data'
RESULTS = ROOT / 'research' / 'transcriptomics' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
manifest = json.loads((DATA / 'run_manifest.json').read_text())
jiang = json.loads((DATA / 'jiang_sources.json').read_text())
for filename, expected in [(Path(manifest['source_url']).name, manifest['source_sha256'])] + [(item['file'], item['sha256']) for item in jiang['files'] if item['file'].endswith('.parquet')]:
    observed = hashlib.sha256((DATA / filename).read_bytes()).hexdigest()
    assert observed == expected, f'Checksum mismatch: {filename}'
print('Verified 3 source data checksums; PIE revision:', jiang['revision'])

Verified 3 source data checksums; PIE revision: d0519f67452a0662c2db30c3831900b38aa3c017


## Processed bulk RNA-seq exposure screen

GSE136998 compares three untreated columns with three columns labeled linalool 8 h at 1× MIC in *Listeria monocytogenes*. The deposited matrix is RSEM FPKM. We calculate a descriptive mean-abundance ratio with a 0.1 pseudocount; no count-likelihood test or FDR is reported. Column multiplicity does not independently verify separate cultures.


In [2]:
expression = pd.read_csv(DATA / 'GSE136998_rsem.genes.fpkm.table.txt.gz', sep='\t', index_col='gene_id')
effects = descriptive_fpkm_ratio(expression, manifest['samples'], control='control', treated='linalool_8h_1xMIC')
effects['direction'] = np.select([effects.descriptive_log2_ratio.gt(0), effects.descriptive_log2_ratio.lt(0)], ['positive', 'negative'], default='zero')
qc = pd.DataFrame({'sample': expression.columns, 'group': [manifest['samples'][sample] for sample in expression.columns], 'median_fpkm': expression.median().values, 'zero_features': expression.eq(0).sum().values, 'nonfinite_values': (~np.isfinite(expression)).sum().values})
effects.to_csv(RESULTS / 'gse136998_descriptive_effects.tsv', sep='\t', index=False)
qc.to_csv(RESULTS / 'gse136998_sample_qc.tsv', sep='\t', index=False)
bulk_summary = {'features': len(effects), 'samples': len(expression.columns), 'directions': effects.direction.value_counts().to_dict(), 'median_fpkm_control_range': [float(qc.loc[qc.group.eq('control'), 'median_fpkm'].min()), float(qc.loc[qc.group.eq('control'), 'median_fpkm'].max())], 'median_fpkm_treated_range': [float(qc.loc[qc.group.ne('control'), 'median_fpkm'].min()), float(qc.loc[qc.group.ne('control'), 'median_fpkm'].max())]}
print(json.dumps(bulk_summary, indent=2))
print(effects.nlargest(3, 'descriptive_log2_ratio')[['feature','control_mean_fpkm','treated_mean_fpkm','descriptive_log2_ratio']].to_string(index=False))

{
  "features": 2976,
  "samples": 6,
  "directions": {
    "positive": 1955,
    "negative": 1011,
    "zero": 10
  },
  "median_fpkm_control_range": [
    20.57,
    22.49
  ],
  "median_fpkm_treated_range": [
    38.349999999999994,
    40.55
  ]
}
feature  control_mean_fpkm  treated_mean_fpkm  descriptive_log2_ratio
 gene57          61.656667           15351.80                7.957607
gene113          19.490000            3744.01                7.578362
gene114           4.583333             815.42                7.444041


## Processed Perturb-seq context comparison

The PIE Jiang A549 IFN-beta and IFN-gamma tables are gene-level CRISPR perturbation summaries, not terpene exposures or installed human variants. We join on exact perturbation and measured feature, restrict to finite log2 effects and control mean expression at least 1 in both contexts, then summarize only the perturbations shared by these two tables. The same target in two contexts is not an independent biological replicate.


In [3]:
beta = pd.read_parquet(DATA / 'a549_ifnb.parquet')
gamma = pd.read_parquet(DATA / 'a549_ifng.parquet')
shared = sorted(set(beta.target) & set(gamma.target))
joined = beta.merge(gamma, on=['target', 'feature'], suffixes=('_ifnb', '_ifng'), validate='one_to_one')
context_rows = []
for target in shared:
    subset = joined.loc[joined.target.eq(target)].copy()
    keep = np.isfinite(subset.log2_fold_change_ifnb) & np.isfinite(subset.log2_fold_change_ifng) & subset.ref_mean_ifnb.ge(1) & subset.ref_mean_ifng.ge(1)
    selected = subset.loc[keep]
    x = selected.log2_fold_change_ifnb.to_numpy()
    y = selected.log2_fold_change_ifng.to_numpy()
    context_rows.append({'target': target, 'shared_tested_features': len(subset), 'features_after_filter': len(selected), 'spearman': float(spearmanr(x,y).statistic), 'sign_agreement': float(np.mean(np.sign(x)==np.sign(y))), 'median_abs_difference': float(np.median(np.abs(x-y))), 'cells_ifnb': int(selected.target_ncells_ifnb.iloc[0]), 'cells_ifng': int(selected.target_ncells_ifng.iloc[0])})
context = pd.DataFrame(context_rows)
context.to_csv(RESULTS / 'a549_context_comparison.tsv', sep='\t', index=False)
print('Input rows:', len(beta), len(gamma), '; shared perturbations:', shared)
print(context.to_string(index=False))

Input rows: 185185 173859 ; shared perturbations: ['BATF2', 'PARP12']
target  shared_tested_features  features_after_filter  spearman  sign_agreement  median_abs_difference  cells_ifnb  cells_ifng
 BATF2                   11574                   2014  0.069027        0.522840               0.097088         605         522
PARP12                   11579                   2014  0.038067        0.514896               0.100983         550         619


## Within-study comparison of defined terpene exposures

GSE138291 has untreated, tea-tree oil, 1,8-cineole, terpinen-4-ol, and a two-compound mixture arms at 2 h with three named columns each. We retrieved GEO's processed FPKM table ([source](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE138nnn/GSE138291/suppl/GSE138291_All_gene_fpkm.list.txt.txt.gz); SHA-256 `2c85b85fdbf973457a8239393bfc797d7e3bc002781b3222f1d63a99cde60ac0`). Sample labels come from the frozen selection manifest. Each arm is compared descriptively with the same untreated columns; these contrasts share a control and are statistically dependent. Oil and mixture arms are shown separately and cannot identify component-specific effects. We report similarity after requiring control mean FPKM ≥1; this is a sensitivity choice, not a hypothesis test.

In [4]:
botrytis_file = DATA / 'GSE138291_All_gene_fpkm.list.txt.txt.gz'
assert hashlib.sha256(botrytis_file.read_bytes()).hexdigest() == '2c85b85fdbf973457a8239393bfc797d7e3bc002781b3222f1d63a99cde60ac0'
botrytis = pd.read_csv(botrytis_file, index_col='#ID')
selected = pd.read_csv(DATA / 'selected_sample_manifest.tsv', sep='\t')
arm_manifest = selected.loc[selected.study.eq('GSE138291')].set_index('label').arm.to_dict()
assert set(botrytis.columns) == set(arm_manifest)
arms = ['1,8-cineole', 'terpinen-4-ol', 'tea tree oil', '1,8-cineole and terpinen-4-ol']
arm_effects = {}
for arm in arms:
    arm_effects[arm] = descriptive_fpkm_ratio(botrytis, arm_manifest, control='untreated', treated=arm).set_index('feature')
control_mean = arm_effects[arms[0]].control_mean_fpkm
expressed = control_mean.ge(1)
effect_matrix = pd.DataFrame({arm: frame.descriptive_log2_ratio for arm, frame in arm_effects.items()})
arm_rows = []
for i, left in enumerate(arms):
    for right in arms[i+1:]:
        x = effect_matrix.loc[expressed, left].to_numpy()
        y = effect_matrix.loc[expressed, right].to_numpy()
        arm_rows.append({'left_arm':left,'right_arm':right,'features_control_fpkm_at_least_1':int(expressed.sum()),'spearman':float(spearmanr(x,y).statistic),'sign_agreement':float(np.mean(np.sign(x)==np.sign(y))),'median_abs_log2_ratio_difference':float(np.median(np.abs(x-y)))})
arm_similarity = pd.DataFrame(arm_rows)
arm_similarity.to_csv(RESULTS / 'gse138291_arm_similarity.tsv', sep='\t', index=False)
effect_matrix.to_csv(RESULTS / 'gse138291_descriptive_ratios.tsv', sep='\t')
print('Processed features:', len(botrytis), '; columns:', len(botrytis.columns), '; control-expressed features:', int(expressed.sum()))
print(arm_similarity.to_string(index=False))
# Sensitivity of the two pure-compound comparison to the abundance offset and shared-control choice.
controls = [sample for sample, arm in arm_manifest.items() if arm == 'untreated']
pure_sensitivity = []
for offset in (0.01, 0.1, 1.0):
    for excluded in [None] + controls:
        kept_controls = [sample for sample in controls if sample != excluded]
        baseline = botrytis[kept_controls].mean(axis=1)
        effects_pair = []
        for arm in arms[:2]:
            treated_samples = [sample for sample, label in arm_manifest.items() if label == arm]
            treated = botrytis[treated_samples].mean(axis=1)
            effects_pair.append(np.log2((treated + offset) / (baseline + offset)))
        selected_pair = baseline.ge(1)
        pure_sensitivity.append({'pseudocount':offset,'excluded_control':excluded or 'none','features':int(selected_pair.sum()),'spearman':float(spearmanr(effects_pair[0][selected_pair],effects_pair[1][selected_pair]).statistic)})
sensitivity = pd.DataFrame(pure_sensitivity)
sensitivity.to_csv(RESULTS / 'gse138291_pure_compound_sensitivity.tsv', sep='\t', index=False)
print('Pure-compound similarity sensitivity:')
print(sensitivity.to_string(index=False))


Processed features: 11456 ; columns: 15 ; control-expressed features: 9000
     left_arm                     right_arm  features_control_fpkm_at_least_1  spearman  sign_agreement  median_abs_log2_ratio_difference
  1,8-cineole                 terpinen-4-ol                              9000  0.432604        0.648444                          0.244595
  1,8-cineole                  tea tree oil                              9000  0.467323        0.667889                          0.240888
  1,8-cineole 1,8-cineole and terpinen-4-ol                              9000  0.384418        0.627778                          0.274249
terpinen-4-ol                  tea tree oil                              9000  0.460448        0.656778                          0.264166
terpinen-4-ol 1,8-cineole and terpinen-4-ol                              9000  0.505599        0.681333                          0.237766
 tea tree oil 1,8-cineole and terpinen-4-ol                              9000  0.324116        0.

Pure-compound similarity sensitivity:
 pseudocount excluded_control  features  spearman
        0.01             none      9000  0.432220
        0.01              T01      8994  0.549504
        0.01              T02      8925  0.501552
        0.01              T03      8818  0.509356
        0.10             none      9000  0.432604
        0.10              T01      8994  0.549965
        0.10              T02      8925  0.502123
        0.10              T03      8818  0.510218
        1.00             none      9000  0.433741
        1.00              T01      8994  0.551652
        1.00              T02      8925  0.504317
        1.00              T03      8818  0.514953


## Study selection and next inferential step

Three selected terpene-exposure cohorts have sample-level selections; one is analyzed above and two remain metadata-only. Their arm counts are useful for planning, but raw reads/counts, reference annotations, culture independence, viability, and chemical identity must be checked before formal modeling. Use the reusable `terputer_bio.transcriptomics` helpers to validate raw count designs and to aggregate guide-assigned single-cell counts by independent sample. Analyze GSE138291 within study first; do not pool its shared-control contrasts as independent. Neither the linalool bulk pilot nor this processed Perturb-seq comparison proves a terpene target.


In [5]:
selected = pd.read_csv(DATA / 'selected_sample_manifest.tsv', sep='\t')
cohorts = selected.groupby(['study','organism','arm'], dropna=False).size().rename('samples').reset_index()
cohorts.to_csv(RESULTS / 'selected_cohort_arms.tsv', sep='\t', index=False)
print(cohorts.to_string(index=False))
summary = {'bulk': bulk_summary, 'perturbseq': {'input_rows_ifnb': len(beta), 'input_rows_ifng': len(gamma), 'shared_perturbations': shared, 'records': context_rows}, 'botrytis_arm_comparisons': arm_rows, 'selected_studies': selected.study.value_counts().to_dict(), 'inference_status': 'descriptive_processed_data_only'}
(RESULTS / 'analysis_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved', RESULTS / 'analysis_summary.json')

      study                                      organism                           arm  samples
  GSE138291                       Botrytis cinerea B05.10                   1,8-cineole        3
  GSE138291                       Botrytis cinerea B05.10 1,8-cineole and terpinen-4-ol        3
  GSE138291                       Botrytis cinerea B05.10                  tea tree oil        3
  GSE138291                       Botrytis cinerea B05.10                 terpinen-4-ol        3
  GSE138291                       Botrytis cinerea B05.10                     untreated        3
PRJNA608617               Penicillium digitatum DSM 62840                R-(+)-limonene        2
PRJNA608617               Penicillium digitatum DSM 62840                     untreated        4
PRJNA913446 Fusarium oxysporum f. sp. radicis-lycopersici                      linalool        3
PRJNA913446 Fusarium oxysporum f. sp. radicis-lycopersici               vehicle control        3
Saved /Users/danielmcshan/GitH

## Interpretation and limits

The bulk processed matrix supports an auditable contrast and sensitivity checks, but no formal differential-expression or pathway inference. The processed Perturb-seq tables allow a bounded context comparison for two shared perturbations; low correlation alone cannot establish context-specific biology when effects are small and cell-level estimates lack independent replicate modeling. The central terpene reproducibility question remains unanswered until compatible independent exposure contrasts are reanalyzed from raw counts/reads. Target dependency requires a chemical-by-genetic intervention design with measured efficacy and viability; target engagement requires separate evidence.
